In [1]:
import warnings
warnings.filterwarnings(action='ignore')

from error_patterns.object_clones import object_clones
from error_patterns.dissociative_object import dissociative_object
from error_patterns.lost_memory import lost_memory
from error_patterns.false_memory import false_memory
from error_patterns.misinformed_objects import misinformed_objects
from error_patterns.transmogrifying_objects import transmogrifying_objects
from localocpa.objects.log.importer.ocel import factory as ocel_import_factory_json
from localocpa.objects.log.importer.csv import factory as ocel_import_factory
from localocpa.algo.util.filtering.log.activity_filtering import filter_infrequent_activities
from localocpa.algo.util.filtering.log.variant_filtering import filter_infrequent_variants

from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

from utils.helpers import get_unique, unlist, get_column_sums, calculate_classification_metrics, detect_outliers_iqr
import pandas as pd
import numpy as np
from itertools import chain

# Import OCED

In [2]:
filename = "sample-logs/bpi2017/BPI2017-Final.csv"
object_types = ["application", "offer"]
parameters = {"obj_names":object_types,
              "val_names":[],
              "act_name":"event_activity",
              "time_name":"event_timestamp",
              "sep":","}
print("Import started")
ocel = ocel_import_factory.apply(file_path= filename,parameters = parameters)
print("Import done")

filtered_ocel = filter_infrequent_activities(ocel, 0.5)
# filtered_ocel = filter_infrequent_variants(filtered_ocel, 0.5)

Import started
Import done


## 1.A. Inject Object Clones

In [4]:
import error_patterns.object_clones  # 모듈 전체 임포트
import importlib
importlib.reload(error_patterns.object_clones) 

<module 'error_patterns.object_clones' from 'c:\\Users\\user\\Desktop\\연구\\OCEL_IF\\object-centric-alignments-main\\error_patterns\\object_clones.py'>

In [7]:
OCEL_polluted = error_patterns.object_clones.object_clones(filtered_ocel,
                              ratio = 0.1,
                              object_root = 'application')

Filtering step 1 . The number of process executions to be filtered:  3151  (ratio:  10.0 %, total process executions:  31509 )


## 1.B. Detect Object Clones

In [ ]:
# Labeling normal/anomalous traces

normal_trace_id = list(chain(*filtered_ocel.variants_dict.values()))
trace_id = list(chain(*OCEL_polluted.variants_dict.values()))
anomalous_trace_id = list(set(trace_id) - set(normal_trace_id) )
detect_vid = []

for id in anomalous_trace_id:
    target_events = list(OCEL_polluted.process_executions[id])
    trace = OCEL_polluted.log.log.loc[OCEL_polluted.log.log['event_id'].isin(target_events)].reset_index(drop = True)
    detect_vid = detect_vid + [trace['application'].apply(lambda x: unlist(x), 1).unique()[0]]

In [ ]:
# Define attributes for similarity check

relevant_attributes = ['event_LoanGoal', 'event_ApplicationType', 'event_RequestedAmount', 'event_CreditScore', 'event_MonthlyCost', 'event_NumberOfTerms'] 

df1 = OCEL_polluted.log.log
df1['application'+'_validation'] = df1['application'].apply(lambda x: x[0] if len(x)>0 else '' )
df1 = df1.loc[df1['application' +'_validation'] != ''].reset_index(drop=True)
df1 = df1[['application'+'_validation']+ relevant_attributes]

table_obj = df1.groupby('application'+'_validation').apply(lambda x: get_unique(x)).reset_index(drop=True)
table_obj['variant'] = table_obj.loc[:, table_obj.columns != 'application'+'_validation'].apply(lambda x: str(list(x)), axis=1)
table_obj_div = table_obj.groupby('variant')['application'+'_validation'].apply(lambda x: list(x))

table_obj_over1 = table_obj_div[table_obj_div.apply(lambda x: len(x)>1)]

# list of similar traces
variant_obj_list = list(chain(*table_obj_over1))

In [ ]:
# Calculate performance of anomaly detection 

actual_positives_set = set(detect_vid)
predicted_positives_set = set(variant_obj_list)

all_applications_in_variants_set = set(table_obj['application_validation'])

tp_count = len(predicted_positives_set.intersection(actual_positives_set))
fp_count = len(predicted_positives_set.difference(actual_positives_set))
fn_count = len(actual_positives_set.difference(predicted_positives_set))

actual_negatives_set = all_applications_in_variants_set.difference(actual_positives_set)
tn_count = len(actual_negatives_set.difference(predicted_positives_set))

if len(detect_vid) > 0:
    Precision = tp_count / (tp_count + fp_count)# sum([1 if i in variant_obj_list else 0 for i in detect_vid]) / len(detect_vid)
else:
    Precision = 0.0 # Or float('nan'), indicating undefined or no positives predicted


if (tp_count + fn_count) > 0:
    Recall = tp_count / (tp_count + fn_count)
else:

    Recall = 0.0

total_population = len(all_applications_in_variants_set) # This is TP + TN + FP + FN
if total_population > 0:
    Accuracy = (tp_count + tn_count) / total_population
else:
    Accuracy = 0.0 # Or float('nan')

if (Precision + Recall) > 0:
    F1_score = 2 * (Precision * Recall) / (Precision + Recall)
else:
    F1_score = 0.0

# --- End of added code ---
# You can now print or use these calculated metrics:
print(f"True Positives (TP): {tp_count}")
print(f"False Positives (FP): {fp_count}")
print(f"True Negatives (TN): {tn_count}")
print(f"False Negatives (FN): {fn_count}")
print(f"Precision: {Precision:.4f}") 
print(f"Recall: {Recall:.4f}")
print(f"Accuracy: {Accuracy:.4f}")
print(f"F1 Score: {F1_score:.4f}")



True Positives (TP): 1591
False Positives (FP): 10655
True Negatives (TN): 22022
False Negatives (FN): 372
True Positives rate (TP): 0.8104941416199695
Precision: 0.1299
Recall: 0.8105
Accuracy: 0.6817
F1 Score: 0.2239


# 2.A. Inject Dissociative Objects

In [335]:
OCEL_polluted = dissociative_object(filtered_ocel,
                              ratio = 0.1,
                              object_root = 'application',
                              relevant_attributes = ['event_LoanGoal', 'event_ApplicationType', 'event_RequestedAmount', 'event_org:resource'])

Filtering step 1 . The number of objects to be filtered:  2924  (ratio:  10.0 %, total objects:  29243 )


## 2.B. Detect Dissociative Objects

In [ ]:
# Get trace id
df_anomaly = pd.DataFrame()
trace_id = list(chain(*OCEL_polluted.variants_dict.values()))
print(len(trace_id))
for id in trace_id:
    target_events = list(OCEL_polluted.process_executions[id])
    trace = OCEL_polluted.log.log.loc[OCEL_polluted.log.log['event_id'].isin(target_events)].reset_index(drop = True)
    trace['trace_id'] = id
    df_anomaly = pd.concat([df_anomaly, trace]).reset_index(drop=True)

28749


In [ ]:
# df_anomaly.to_csv('df_polluted_dissociative.csv', index=False)

In [7]:
df_anomaly = pd.read_csv('df_polluted_dissociative.csv')
pipe = Pipeline(
    steps=[
        ("IndexBasedTransformer", IndexBasedTransformer(case_id_col = 'trace_id',
                        cat_cols = ['event_activity'],
                        num_cols = []))
    ]
)
X = pipe.fit_transform(df_anomaly[['event_activity', 'trace_id']])
X.shape

(28749, 167)

In [ ]:
# Encoding to numerical matrices

from utils.IndexBasedTransformer import IndexBasedTransformer
from sklearn.pipeline import Pipeline

pipe = Pipeline(
    steps=[
        ("IndexBasedTransformer", IndexBasedTransformer(case_id_col = 'trace_id',
                        cat_cols = [OCEL_polluted.parameters['act_name']],
                        num_cols = []))
    ]
)
X = pipe.fit_transform(df_anomaly[[OCEL_polluted.parameters['act_name'], 'trace_id']])
X.shape

In [11]:
# 1. Moore-Penrose inv

X = X*1
X_pinv = np.linalg.pinv(X.T @ X)
X2 = X.T.reset_index(drop=True)
H = X @ X_pinv @ X2

del X_pinv
del X2
leverage =  np.diag(H)  

# leverage =  get_column_sums(H)  

# Calculate length and normalize
length = np.sum(X, axis=1)  # Equivalent to apply(x, 1, sum) in R
z_norm = (length - np.mean(length)) / np.std(length)

# Sigmoid function
sigmoid_leng = 1 / (1 + np.exp(-z_norm))

# Weighted leverage calculation
threshold = -2.2822 + np.max(length)**0.3422
if threshold < 0 or len(np.unique(length)) == 1:
    leverage = leverage
else:
    leverage = leverage * (1 - sigmoid_leng)**(-threshold)

In [ ]:

temp = df_anomaly[['trace_id' ,'label']]
temp.loc[temp.label !='', ['label']]  = 1
temp.loc[temp.label == '', ['label']]  = 0
temp = temp.drop_duplicates().reset_index(drop=True)
temp = temp.groupby(['trace_id'], as_index=False)['label'].apply(sum)
temp.loc[temp.label >0 , ['label']] = 1
result = pd.DataFrame({'trace_id':X.index, 'anomaly_score':list(leverage)}) 
result = result.merge(  temp, how = 'left', on = 'trace_id')

result['pred'] = (result['anomaly_score'] > np.percentile(result['anomaly_score'], 85))*1
results = calculate_classification_metrics(result, pred_col='pred', label_col='label')

# Print results
print("--- Classification Metrics ---")
print(f"True Positives (TP): {results['TP']}")
print(f"True Negatives (TN): {results['TN']}")
print(f"False Positives (FP): {results['FP']}")
print(f"False Negatives (FN): {results['FN']}")
print(f"Precision: {results['Precision']:.4f}")
print(f"Recall: {results['Recall']:.4f}")
print(f"Accuracy: {results['Accuracy']:.4f}")
print(f"F1-score: {results['F1-score']:.4f}")

--- Classification Metrics ---
True Positives (TP): 2401
True Negatives (TN): 24229
False Positives (FP): 1846
False Negatives (FN): 273
Precision: 0.5653
Recall: 0.8979
Accuracy: 0.9263
F1-score: 0.6938


## 3.A. Inject Misinformed Objects

In [298]:
OCEL_polluted = misinformed_objects(filtered_ocel,
                              ratio = 0.1,
                              related_attributes = ['event_OfferedAmount', 'event_MonthlyCost'],  # ['event_OfferedAmount', 'event_MonthlyCost']
                              level = 'event')

Correlation analysis between ['event_OfferedAmount', 'event_MonthlyCost'] : PearsonRResult(statistic=0.718711018809373, pvalue=0.0)
Filtering step 1 . The number of process executions to be filtered:  3151  (ratio:  10.0 %, total process executions:  31509 )


## 3.B. Detect Misinformed Objects

In [299]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

x_col = 'event_OfferedAmount'
y_col = 'event_MonthlyCost'
df1 = OCEL_polluted.log.log[['event_OfferedAmount', 'event_MonthlyCost',  'label']]
df1 = df1.dropna()
df1 = df1.reset_index(drop = True)

X_train = df1[[x_col]].values.reshape(-1, 1)
y_train = df1[y_col].values
X_train= X_train.astype(np.float64)
# 3. Train the Linear Regression Model
model = LinearRegression()
model.fit(X_train, y_train)

# 4. Predictions on the test set
y_pred = model.predict(X_train)

# 5. Calculate Prediction Errors and Confidence Intervals

# Calculate residuals (prediction errors)
residuals = y_train - y_pred

# Calculate the standard deviation of the residuals
std_residuals = np.std(residuals)

# Calculate the mean squared error (MSE)
mse = mean_squared_error(y_train, y_pred)
print(f"Mean Squared Error: {mse}")

# Calculate a confidence interval for the predictions
# For a 95% confidence interval, use 1.96 * std_residuals
confidence_interval = 1.65 * std_residuals

# Calculate the upper and lower bounds of the confidence interval
upper_bound = y_pred + confidence_interval
lower_bound = y_pred - confidence_interval

df1['upper'] = upper_bound
df1['lower'] = lower_bound
df1['pred'] = y_pred
df2 = df1.loc[df1.label != '']

Mean Squared Error: 34645.96896538774


In [ ]:
df1.loc[df1.label != '', 'label'] = 1
df1.loc[df1.label == '', 'label'] = 0
df1['pred'] =0
df1.loc[(df1['event_MonthlyCost'].astype(np.float64) > df1['upper']) | (df1['event_MonthlyCost'].astype(np.float64) < df1['lower']), 'pred'] = 1

results = calculate_classification_metrics(df1, pred_col='pred', label_col='label')

# Print results
print("--- Classification Metrics ---")
print(f"True Positives (TP): {results['TP']}")
print(f"True Negatives (TN): {results['TN']}")
print(f"False Positives (FP): {results['FP']}")
print(f"False Negatives (FN): {results['FN']}")
print(f"Precision: {results['Precision']:.4f}")
print(f"Recall: {results['Recall']:.4f}")
print(f"Accuracy: {results['Accuracy']:.4f}")
print(f"F1-score: {results['F1-score']:.4f}")

## 4.A. Transmogrifying Objects

In [312]:
import error_patterns.transmogrifying_objects  # 모듈 전체 임포트
import importlib
importlib.reload(error_patterns.transmogrifying_objects) 

<module 'error_patterns.transmogrifying_objects' from 'c:\\Users\\user\\Desktop\\연구\\OCEL_IF\\object-centric-alignments-main\\error_patterns\\transmogrifying_objects.py'>

In [ ]:
OCEL_polluted = error_patterns.transmogrifying_objects.transmogrifying_objects(filtered_ocel,
                              ratio = 0.1,
                              constant_attribute = 'event_RequestedAmount')

Filtering step 1 . The number of process executions to be filtered:  3151  (ratio:  10.0 %, total process executions:  31509 )


## 4.B. Detect Transmogrifying Objects

In [316]:
variant_id = list(OCEL_polluted.variants_dict.keys())
list_TP = []
label = []
i = 0
for id in variant_id:
    target_events = list(OCEL_polluted.process_executions[OCEL_polluted.variants_dict[id][0]])
    trace = OCEL_polluted.log.log.loc[OCEL_polluted.log.log['event_id'].isin(target_events)].reset_index(drop = True)
    outliers = detect_outliers_iqr(trace['event_RequestedAmount'])
    list_TP = list_TP + [sum(outliers) >0]
    label = label + [sum( (trace.label !=''))>0]
    i= i+1

res =pd.DataFrame({'v_id':variant_id, 'TP': list_TP, 'label':label })


In [317]:
results = calculate_classification_metrics(res, pred_col='TP', label_col='label')

# Print results
print("--- Classification Metrics ---")
print(f"True Positives (TP): {results['TP']}")
print(f"True Negatives (TN): {results['TN']}")
print(f"False Positives (FP): {results['FP']}")
print(f"False Negatives (FN): {results['FN']}")
print(f"Precision: {results['Precision']:.4f}")
print(f"Recall: {results['Recall']:.4f}")
print(f"Accuracy: {results['Accuracy']:.4f}")
print(f"F1-score: {results['F1-score']:.4f}")

--- Classification Metrics ---
True Positives (TP): 152
True Negatives (TN): 545
False Positives (FP): 0
False Negatives (FN): 20
Precision: 1.0000
Recall: 0.8837
Accuracy: 0.9721
F1-score: 0.9383


## 5,6. Demonstrate Lost Relative and Cuckoo's Egg in BPI2020

In [ ]:
import pm4py
dat1 = pm4py.read_xes("sample-logs/bpi2020/PermitLog.xes.gz")
dat2 = pm4py.read_xes("sample-logs/bpi2020/PrepaidTravelCost.xes.gz")

dat1 = pm4py.convert_to_dataframe(dat1)
dat2 = pm4py.convert_to_dataframe(dat2)

parsing log, completed traces :: 100%|██████████| 7065/7065 [00:01<00:00, 3669.12it/s]


In [ ]:
dat1['request'] = dat1['case:travel permit number'].apply(lambda x: x.split(' ')[-1])
dat1['permit'] = dat1['case:concept:name'].apply(lambda x: x.split(' ')[-1])

dat2['request'] = dat2['case:RfpNumber'].apply(lambda x: x.split(' ')[-1])
dat2['permit'] = dat2['case:Permit id'].apply(lambda x: x.split(' ')[-1])

In [ ]:
cor = dat1[['request', 'permit']]
cor = cor.drop_duplicates()

cor2 = dat2[['request', 'permit']]
cor2 = cor2.drop_duplicates()

print("Total object relations in event data:",  len(cor2))

In [ ]:
filter1 = cor2.loc[cor2['permit'].isin(cor['permit'])].reset_index(drop=True)
filter2 = filter1.loc[filter1['request'].isin(cor['request'])]
filter3 = filter1.loc[~filter1['request'].isin(cor['request'])]

print("# of object relations shared in both object table and event data:",  len(filter2)) 
print("# of object relations with Lost Relative:",  len(cor2)-len(filter1)) 
print("# of object relations with Cuckoo's Egg:",  len(filter3)) 

## 7.A. Inject Lost Memory

In [318]:
OCEL_polluted = lost_memory(filtered_ocel,
                              ratio = 0.1,
                              related_objects = ['application', 'offer'])

Filtering step 1 . The number of object relations to be perturbed by lost-memory:  3151  (ratio:  10.0 %, total object relations:  31509 )


## 7.B. Detect Lost Memory (using regression)

In [ ]:
temp_df = filtered_ocel.log.log
x_col = ['event_activity' , 'event_LoanGoal', 'event_ApplicationType', 'event_RequestedAmount', 'event_org:resource']

y_train = temp_df['offer'].apply(len).astype(str).tolist()

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  # sparse=False for compatibility
encoder.fit(temp_df[x_col])
encoded_x = encoder.transform(temp_df[x_col])
feature = encoder.get_feature_names_out(x_col)
X_train = pd.DataFrame(encoded_x, columns=encoder.get_feature_names_out(x_col))  # More descriptive column names

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [ ]:
test_df = OCEL_polluted.log.log
y_test = test_df['offer'].apply(len).tolist()

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  # sparse=False for compatibility
encoder.fit(test_df[x_col])
encoded_x = encoder.transform(test_df[x_col])
feature = encoder.get_feature_names_out(x_col)
X_test = pd.DataFrame(encoded_x, columns=encoder.get_feature_names_out(x_col))  # More descriptive column names

y_pred = model.predict(X_test)
y_pred =  (y_pred > 0.5)

In [324]:

res = pd.DataFrame({'pred':y_pred , 'label': y_test})
results = calculate_classification_metrics(res, pred_col='pred', label_col='label')

# Print results
print("--- Classification Metrics ---")
print(f"True Positives (TP): {results['TP']}")
print(f"True Negatives (TN): {results['TN']}")
print(f"False Positives (FP): {results['FP']}")
print(f"False Negatives (FN): {results['FN']}")
print(f"Precision: {results['Precision']:.4f}")
print(f"Recall: {results['Recall']:.4f}")
print(f"Accuracy: {results['Accuracy']:.4f}")
print(f"F1-score: {results['F1-score']:.4f}")

--- Classification Metrics ---
True Positives (TP): 108724
True Negatives (TN): 102462
False Positives (FP): 5461
False Negatives (FN): 0
Precision: 0.9522
Recall: 1.0000
Accuracy: 0.9748
F1-score: 0.9755


# 8.A. Inject False Memory

In [214]:
import error_patterns.false_memory  # 모듈 전체 임포트
import importlib
importlib.reload(error_patterns.false_memory) 

<module 'error_patterns.false_memory' from 'c:\\Users\\user\\Desktop\\연구\\OCEL_IF\\object-centric-alignments-main\\error_patterns\\false_memory.py'>

In [117]:
OCEL_polluted = false_memory(filtered_ocel,
                              ratio = 0.1,
                              related_objects = ['application', 'offer'])

# TBD: split target object and related objects
# TBD: in case that there are 3 or more related objects

Filtering step 1 . The number of object relations to be perturbed by false-memory:  3151  (ratio:  10.0 %, total object relations:  31509 )


## 8.B. Detect False Memory

In [119]:
from itertools import chain

# Get trace id
df_anomaly = pd.DataFrame()
trace_id = list(chain(*OCEL_polluted.variants_dict.values()))
print(len(trace_id))
for id in trace_id:
    target_events = list(OCEL_polluted.process_executions[id])
    trace = OCEL_polluted.log.log.loc[OCEL_polluted.log.log['event_id'].isin(target_events)].reset_index(drop = True)
    trace['trace_id'] = id
    df_anomaly = pd.concat([df_anomaly, trace]).reset_index(drop=True)

31365


In [ ]:
# df_anomaly.to_csv('df_polluted_FALSE_MEMORY.csv', index=False)

In [326]:
from utils.IndexBasedTransformer import IndexBasedTransformer
from sklearn.pipeline import Pipeline

pipe = Pipeline(
    steps=[
        ("IndexBasedTransformer", IndexBasedTransformer(case_id_col = 'trace_id',
                        cat_cols = [OCEL_polluted.parameters['act_name']],
                        num_cols = []))
    ]
)
X = pipe.fit_transform(df_anomaly[[OCEL_polluted.parameters['act_name'], 'trace_id']])
X.shape

(31365, 147)

In [ ]:
# 1. Moore-Penrose inv

X = X*1
X_pinv = np.linalg.pinv(X.T @ X)
X2 = X.T.reset_index(drop=True)
H = X @ X_pinv @ X2


del X_pinv
del X2
leverage =  np.diag(H)  

# Calculate length and normalize
length = np.sum(X, axis=1)  # Equivalent to apply(x, 1, sum) in R
z_norm = (length - np.mean(length)) / np.std(length)

# Sigmoid function
sigmoid_leng = 1 / (1 + np.exp(-z_norm))

# Weighted leverage calculation
threshold = -2.2822 + np.max(length)**0.3422
if threshold < 0 or len(np.unique(length)) == 1:
    leverage = leverage
else:
    leverage = leverage * (1 - sigmoid_leng)**(-threshold)

In [ ]:
temp = df_anomaly[['trace_id' ,'label']]
temp.loc[temp.label.isna(), ['label']]  = 0
temp.loc[temp.label !=0, ['label']]  = 1

temp = temp.drop_duplicates().reset_index(drop=True)
temp = temp.groupby(['trace_id'], as_index=False)['label'].apply(sum)
temp.loc[temp.label >0 , ['label']] = 1
result = pd.DataFrame({'trace_id':X.index, 'anomaly_score':list(leverage)}) 
result = result.merge(  temp, how = 'left', on = 'trace_id')
result['pred'] = (result['anomaly_score'] > np.percentile(result['anomaly_score'], 85))*1

res = pd.DataFrame({'pred':y_pred , 'label': y_test})
results = calculate_classification_metrics(result, pred_col='pred', label_col='label')

# Print results
print("--- Classification Metrics ---")
print(f"True Positives (TP): {results['TP']}")
print(f"True Negatives (TN): {results['TN']}")
print(f"False Positives (FP): {results['FP']}")
print(f"False Negatives (FN): {results['FN']}")
print(f"Precision: {results['Precision']:.4f}")
print(f"Recall: {results['Recall']:.4f}")
print(f"Accuracy: {results['Accuracy']:.4f}")
print(f"F1-score: {results['F1-score']:.4f}")

--- Classification Metrics ---
True Positives (TP): 2938
True Negatives (TN): 26638
False Positives (FP): 1751
False Negatives (FN): 38
Precision: 0.6266
Recall: 0.9872
Accuracy: 0.9430
F1-score: 0.7666
